# Interpolation and Approximation

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 06.04 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/05_interpolation_and_approximation.ipynb)

---

## 🎯 Learning Objective

Study polynomial interpolation and splines, and connect them to the universal approximation theorem.

---

## 📐 Theory

Given a handful of $(x_i, y_i)$ pairs, **interpolation** builds a function passing through all
of them exactly; **approximation** relaxes that requirement to get an overall better fit. Both
questions turn out to connect directly to why neural networks work at all as function fitters.

### Polynomial interpolation

Given $n+1$ points, there is exactly **one** polynomial of degree $\le n$ passing through all
of them (the Lagrange interpolation formula constructs it explicitly). This sounds like a
complete solution to interpolation — until you look at what happens *between* the given
points, not just at them.

### Runge's phenomenon

You might expect more data points (a higher-degree polynomial) to always fit better. For
evenly-spaced points, this is often false: **Runge's phenomenon** is the tendency of
high-degree polynomial interpolants to oscillate wildly near the edges of the interval, even
though they match every given point exactly. The polynomial has enough "wiggle room" (many
degrees of freedom) to satisfy every constraint while still behaving badly in between — a
single global polynomial is a surprisingly poor tool for smooth, well-behaved local fitting.

### Splines: piecewise polynomials

A **spline** fits a *separate*, low-degree polynomial (commonly cubic) on each sub-interval
between consecutive points, then stitches the pieces together enforcing continuity of value,
first derivative, and (for cubic splines) second derivative at each joint. This local
flexibility — each piece only has to behave well over its own small interval — avoids Runge's
phenomenon entirely, at the cost of the fit no longer being expressible as a single simple
formula. This "many simple local pieces rather than one complicated global rule" idea reappears
throughout numerical computing and, as this notebook's AI section argues, in how neural
networks approximate functions too.

### The Universal Approximation Theorem

The **Universal Approximation Theorem** (Cybenko 1989; Hornik et al. 1989) states that a
feedforward network with a single hidden layer, a non-polynomial activation function, and
enough hidden units can approximate *any* continuous function on a compact domain to arbitrary
accuracy. This is an *existence* result, not a construction — it guarantees enough capacity
exists, but says nothing about whether gradient descent (`04.02`) will actually *find* the
right weights, or how many hidden units "enough" really means in practice. The theorem's proof
strategy has a flavor similar to splines: a sum of many simple, localized basis functions
(here, shifted/scaled sigmoids or ReLUs) can approximate a complicated global function, echoing
the "combine simple local pieces" idea from splines above.

### Basis functions and positional encoding

More generally, approximating a function as a weighted sum of fixed **basis functions**
$f(x) \approx \sum_k c_k \phi_k(x)$ is a unifying idea behind polynomial interpolation
($\phi_k(x)=x^k$), Fourier series ($\phi_k(x) = \sin(kx), \cos(kx)$), and — as the AI section
shows concretely — the sinusoidal **positional encodings** used in the original Transformer
architecture, which represent a token's position as a vector of sines and cosines at different
frequencies, precisely a fixed Fourier-style basis rather than a learned one.

---

In [ ]:
# Setup
import numpy as np
from scipy.interpolate import CubicSpline
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Fitting a single high-degree polynomial through evenly-spaced points and a cubic spline
through the same points, both matching every point exactly, shows a striking difference
*between* the points: the polynomial oscillates wildly near the edges (Runge's phenomenon).

In [ ]:
# Runge's phenomenon: a single high-degree polynomial through evenly-spaced points can
# oscillate wildly near the edges, even though it matches every node exactly.
f = lambda x: 1 / (1 + 25 * x**2)
x_nodes = np.linspace(-1, 1, 15)
y_nodes = f(x_nodes)
x_fine = np.linspace(-1, 1, 300)

poly_coefs = np.polyfit(x_nodes, y_nodes, deg=14)   # exactly interpolates all 15 points
y_poly = np.polyval(poly_coefs, x_fine)
y_spline = CubicSpline(x_nodes, y_nodes)(x_fine)
y_true = f(x_fine)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(x_fine, y_true, color="k", lw=1.5, ls=":", label="true function")
ax.plot(x_fine, y_poly, color="#C44E52", lw=2, label="degree-14 polynomial (exact at nodes)")
ax.plot(x_fine, y_spline, color="#4C72B0", lw=2, label="cubic spline (also exact at nodes)")
ax.plot(x_nodes, y_nodes, 'ko', markersize=5, label="interpolation nodes")
ax.set_ylim(-1, 2)
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
ax.set_title("Same nodes, same exact fit AT the nodes --\nwild oscillation BETWEEN them for high-degree polynomials")
ax.legend(fontsize=9)
plt.show()

print(f"Max error, degree-14 polynomial: {np.max(np.abs(y_poly - y_true)):.4f}")
print(f"Max error, cubic spline:        {np.max(np.abs(y_spline - y_true)):.4f}")
print(f"Both pass through EVERY node exactly, but the spline is dramatically closer everywhere else.")

## 🐍 Implementation from Scratch

We build a natural cubic spline from scratch by solving the standard tridiagonal system for
each node's second derivative, verify it matches `scipy.interpolate.CubicSpline` exactly, and
confirm the universal approximation theorem's claim empirically: does more hidden-layer
capacity actually reduce approximation error on a fixed target function?

In [ ]:
def natural_cubic_spline_coeffs(x, y):
    """Solve the standard tridiagonal system for each node's second derivative M_i, with
    natural boundary conditions (M_0 = M_n = 0 -- the spline flattens out at both ends)."""
    n = len(x) - 1
    h = np.diff(x)
    A = np.zeros((n + 1, n + 1))
    rhs = np.zeros(n + 1)
    A[0, 0] = 1          # natural BC: M_0 = 0
    A[n, n] = 1          # natural BC: M_n = 0
    for i in range(1, n):
        A[i, i - 1] = h[i - 1]
        A[i, i] = 2 * (h[i - 1] + h[i])
        A[i, i + 1] = h[i]
        rhs[i] = 6 * ((y[i + 1] - y[i]) / h[i] - (y[i] - y[i - 1]) / h[i - 1])
    return np.linalg.solve(A, rhs)

def eval_natural_spline(x_nodes, y_nodes, M, x_query):
    idx = np.clip(np.searchsorted(x_nodes, x_query) - 1, 0, len(x_nodes) - 2)
    x0, x1 = x_nodes[idx], x_nodes[idx + 1]
    y0, y1 = y_nodes[idx], y_nodes[idx + 1]
    M0, M1 = M[idx], M[idx + 1]
    h = x1 - x0
    # Standard cubic-piece formula for a natural spline, built directly from each piece's
    # second derivatives at its two endpoints (Burden & Faires' construction)
    return (M0 / (6 * h) * (x1 - x_query) ** 3 + M1 / (6 * h) * (x_query - x0) ** 3
            + (y0 / h - M0 * h / 6) * (x1 - x_query) + (y1 / h - M1 * h / 6) * (x_query - x0))

x_nodes = np.linspace(0, 10, 8)
y_nodes = np.sin(x_nodes) + 0.1 * x_nodes
M = natural_cubic_spline_coeffs(x_nodes, y_nodes)
x_query = np.linspace(0, 10, 100)
y_ours = eval_natural_spline(x_nodes, y_nodes, M, x_query)
y_scipy = CubicSpline(x_nodes, y_nodes, bc_type='natural')(x_query)
print(f"Our spline vs scipy.interpolate.CubicSpline: max diff = {np.max(np.abs(y_ours - y_scipy)):.2e}")
print(f"Match: {np.allclose(y_ours, y_scipy, atol=1e-8)}")

# --- Universal Approximation Theorem, tested empirically: does more capacity help? ---
f_true = lambda x: np.sin(3 * x) + 0.3 * x**2
x_train = np.linspace(-3, 3, 200)
y_train = f_true(x_train)
Xt = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
yt = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

def train_and_eval(n_hidden, seed, steps=3000, lr=0.02):
    torch.manual_seed(seed)
    model = torch.nn.Sequential(torch.nn.Linear(1, n_hidden), torch.nn.Tanh(), torch.nn.Linear(n_hidden, 1))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = torch.nn.functional.mse_loss(model(Xt), yt)
        loss.backward()
        opt.step()
    return loss.item()

print(f"\n{'hidden units':>12} {'mean final MSE (3 seeds)':>26}")
for n_hidden in [2, 4, 8, 16, 32, 64]:
    losses = [train_and_eval(n_hidden, seed=s) for s in [1, 2, 3]]
    print(f"{n_hidden:12d} {np.mean(losses):26.6f}")
print("\nError shrinks (with some optimization noise per individual run) as capacity grows --")
print("consistent with the Universal Approximation Theorem's existence claim, though the")
print("theorem itself says nothing about whether gradient descent finds the best weights.")

## 🤖 Why This Matters for AI

The Universal Approximation Theorem is the formal justification for calling neural networks
"universal function approximators" — the property this whole repo's approach relies on
implicitly every time a small network is trained to fit some target function in these
notebooks. A more concrete, literal use of approximation theory shows up in the original
Transformer's **sinusoidal positional encoding**: each position is represented as a fixed
vector of $\sin$/$\cos$ values at geometrically spaced frequencies — a **fixed Fourier-style
basis**, exactly the $\sum_k c_k\phi_k(x)$ idea from the Theory section, with $\phi_k$ fixed in
advance (unlike a network's learned basis functions) and $c_k$ implicitly determined by which
frequencies the attention mechanism ends up using. Below, we build this positional encoding
from its published formula and inspect how different dimensions oscillate at very different
rates — encoding both fine-grained (nearby-position) and coarse-grained (far-position)
structure simultaneously.

In [ ]:
# Sinusoidal positional encoding, exactly as specified in "Attention Is All You Need"
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))  # geometrically decreasing frequency per dim-pair
    angles = pos * angle_rates
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles[:, 0::2])   # even dims: sine
    pe[:, 1::2] = np.cos(angles[:, 1::2])   # odd dims: cosine, same frequency as its sine partner
    return pe

seq_len, d_model = 100, 16
pe = positional_encoding(seq_len, d_model)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
im = axes[0].imshow(pe.T, cmap="RdBu", aspect="auto")
axes[0].set_xlabel("position in sequence"); axes[0].set_ylabel("encoding dimension")
axes[0].set_title("Positional encoding: each row is a fixed basis\nfunction, evaluated at every position")
fig.colorbar(im, ax=axes[0])

axes[1].plot(pe[:, 0], color="#C44E52", label="dimension 0 (fast oscillation)")
axes[1].plot(pe[:, 14], color="#4C72B0", label="dimension 14 (slow oscillation)")
axes[1].set_xlabel("position in sequence"); axes[1].set_ylabel("encoding value")
axes[1].set_title("Low dimensions vary quickly (fine position detail);\nhigh dimensions vary slowly (coarse detail)")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Positional encoding shape: {pe.shape} (seq_len={seq_len}, d_model={d_model})")
print(f"Dimension 0 period (positions per full cycle): ~{2*np.pi:.2f}")
print(f"Dimension 14 period: ~{2*np.pi / (1/10000**(14/d_model)):.1f}")
print("Just like a spline's local pieces or a Fourier series' harmonics, different dimensions")
print("here are different FIXED basis functions -- the model learns how much to weight each one.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For 4 nodes at $x=(0,1,2,3)$ with $y=(0,1,0,1)$, set up by hand the tridiagonal system
   `natural_cubic_spline_coeffs` would solve for the second-derivative values $M_0,\ldots,M_3$,
   then solve it. Check $M_0=M_3=0$ (the natural boundary condition) and confirm your values
   against `CubicSpline(x, y, bc_type='natural')`'s second derivative at the nodes.

<details>
<summary>💡 Solution</summary>

With $h_0=h_1=h_2=1$, the interior rows give $M_0+4M_1+M_2=6((1-1)/1)-6((1-0)/1))=-12$ and
$M_1+4M_2+M_3=6((1-0)/1)-6((0-1)/1))=12$, with $M_0=M_3=0$ from the natural boundary rows. Solving
gives $M=(0,-4,4,0)$, matching `CubicSpline`'s second derivative at the nodes exactly (up to
floating-point rounding, $\sim10^{-16}$).

</details>

### 💭 UNDERSTAND
2. Increasing the number of *evenly-spaced* interpolation nodes from 15 to 25 makes a
   degree-14 polynomial's worst-case error on the Runge function get dramatically *worse*
   (not better), while a cubic spline's error stays small and even improves slightly. Why does
   adding more data points make one method worse and the other better?

<details>
<summary>💡 Solution</summary>

Numerically, the degree-14 polynomial's max error grows from about $7.19$ (15 nodes) to about
$257.2$ (25 nodes, now degree 24) — more than $35\times$ worse — while the cubic spline's max
error stays around $0.002$–$0.003$ throughout. A single global polynomial of degree $n-1$ through
$n$ evenly-spaced points has $n-1$ derivatives that must all vanish somewhere to satisfy the
interpolation constraints exactly, and evenly-spaced nodes are provably close to the worst
possible choice for controlling a global polynomial's behavior between nodes (Chebyshev nodes,
clustered near the endpoints, fix this specific problem) — so more evenly-spaced nodes only give
the polynomial more freedom to oscillate wildly near the edges. A spline never has this problem
because each piece only has to fit *its own* short interval; adding more nodes just adds more
(still well-behaved) short pieces, never forcing any single piece to satisfy far-away constraints.

</details>

### ✏️ DERIVE
3. The Lagrange interpolation basis functions $L_j(x) = \prod_{m\ne j}\frac{x-x_m}{x_j-x_m}$
   satisfy $\sum_j L_j(x) = 1$ for every $x$ (not just at the nodes) — the "partition of unity"
   property. Prove this using the fact that the constant function $g(x)=1$ is itself a
   polynomial of degree $\le n-1$ agreeing with $y_j=1$ at every node, combined with the
   uniqueness of polynomial interpolation.

<details>
<summary>💡 Solution outline</summary>

The Lagrange interpolant of any data $(x_j, y_j)$ is $p(x) = \sum_j y_j L_j(x)$ by construction
(each $L_j$ is $1$ at $x_j$ and $0$ at every other node). Take $y_j=1$ for every $j$: the
interpolant of this data is $\sum_j L_j(x)$. But the constant function $g(x)\equiv 1$ is a
polynomial of degree $0\le n-1$ that *also* passes through every $(x_j, 1)$ exactly. Since
polynomial interpolation of $n$ points by a degree-$\le(n-1)$ polynomial is unique (stated in
the Theory section), these two must be the *same* polynomial: $\sum_j L_j(x) = 1$ for all $x$,
not merely at the nodes. Numerically, evaluating $\sum_j L_j(x)$ at several non-node test points
for a 4-node example confirms the sum is exactly $1.0$ everywhere tested.

</details>

### 🐍 IMPLEMENT
4. `train_and_eval` in the Implementation cell trains a 1-hidden-layer network and reports
   final MSE for varying hidden-unit counts. Extend it to also report, for each hidden-unit
   count, the *variance* of the final loss across several random seeds (not just the mean).
   Does variance across seeds shrink, grow, or stay flat as capacity increases?

<details>
<summary>✅ How to know you're right</summary>

Run with at least 5 seeds per hidden-unit count (more than the notebook's 3, for a more reliable
variance estimate) and confirm your reported numbers are the sample variance (or standard
deviation) of the per-seed final losses, not of anything else. There's no single "correct"
qualitative trend the exercise is grading you against — different seeds/architectures can go
either way — but a correct implementation should show the *same* mean-MSE-decreases-with-capacity
trend the notebook already established, confirming your extended version didn't silently change
what's being measured.

</details>

### 🤖 APPLY
5. `06.04`'s conjugate gradient exercises discussed how a matrix's structure (SPD, condition
   number) determines an iterative solver's convergence behavior. Draw the analogous structural
   argument here: explain why a $B$-spline basis (each basis function nonzero only over a few
   neighboring intervals) makes fitting a spline to $n$ points a **sparse** linear system,
   while the direct Lagrange/monomial polynomial basis used in `np.polyfit` gives a **dense**
   system of the same size — and why this matters computationally for a spline fit to, say,
   $n=10^5$ points (as might appear in fitting a smooth learned function to a large lookup
   table).

<details>
<summary>✅ What you should observe</summary>

A cubic spline's coefficient system (like `natural_cubic_spline_coeffs`'s tridiagonal matrix) is
sparse because each interior equation only involves the current node and its two immediate
neighbors — a direct consequence of splines being *local* (each piece's polynomial only
depends on nearby data). A monomial or Lagrange basis has no such locality: every basis function
is nonzero across the whole domain, so the linear system relating coefficients to data values is
generally dense. For $n=10^5$, solving a dense $n\times n$ system directly is computationally
infeasible ($O(n^3)$ time, $O(n^2)$ storage — recall `06.04`'s framing), while the tridiagonal
spline system solves in $O(n)$ time with $O(n)$ storage using a specialized tridiagonal solver.
This is the same "exploit structure rather than treat every system as generic dense linear
algebra" theme as `06.04`'s iterative-solver discussion, just showing up via basis choice instead
of iteration count.

</details>

### 🚀 CHALLENGE
6. Modify `positional_encoding` to use base $100$ instead of $10000$ in the frequency formula,
   and compare the range of periods spanned across the 16 dimensions in each case. Explain, in
   terms of a transformer's maximum useful context length, why the original paper's choice of
   $10000$ specifically (rather than, say, $100$) might matter.

<details>
<summary>🔍 What a strong answer covers</summary>

With base $10000$ and $d_{model}=16$, the fastest dimension (dim 0) has period $2\pi\approx6.28$
positions and the slowest (dim 14) has period $\approx19869$ positions — a span of roughly four
orders of magnitude. Switching to base $100$ keeps the fastest period the same ($2\pi$, since
that dimension's exponent is $0$ regardless of base) but shrinks the slowest period to only
$\approx353$ positions — a span of about two orders of magnitude, half as many as before. A
strong answer connects this directly to context length: the *slowest*-varying dimensions are
what let the model distinguish positions that are far apart (e.g. position 50 vs. position 550 in
a long document), since fast-varying dimensions alone repeat every few positions and can't tell
distant positions apart without ambiguity. A smaller base compresses the maximum distinguishable
range, so a transformer meant to handle long contexts (thousands of tokens) needs its slowest
dimension's period to comfortably exceed the longest sequence length it will ever see — which is
exactly why $10000$ (giving periods up into the tens of thousands) was chosen over a much smaller
base for a model expected to handle sequences in that length range.

</details>

---

## 📚 Further Reading
- Cybenko, ["Approximation by Superpositions of a Sigmoidal Function"](https://link.springer.com/article/10.1007/BF02551274) (1989)
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) (Section 3.5, Positional Encoding)
- Burden & Faires, *Numerical Analysis*, Ch. 3 (Interpolation and Polynomial Approximation)

---

*Next notebook: [Mixed Precision Training](06_mixed_precision_training.ipynb)*